# 내 물건으로 Stable Diffusion 미세조정하기 (SD 1.5 DreamBooth LoRA)

이 노트북은 위에서 아래로 셀을 차례로 실행하면 끝난다. 무료 T4 GPU 기준 20~30분이 걸린다.

| 단계 | 내용 | 시간 (예상) |
|---|---|---|
| 1 | 패키지 설치, 모델 다운로드 시작 | 2~3분 |
| 2 | 학습 대상 고르기 (샘플 또는 내 사진) | 1분 |
| 3 | **LoRA 직접 학습** | 10~15분 |
| 4 | 네 방법 비교: 베이스 / LoRA / Textual Inversion / IP-Adapter | 5분 |
| 5 | LoRA 세기 조절, 자유 생성 | 남는 시간 |

**시작 전에 GPU를 켠다.**
- Colab: 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`
- Kaggle: 오른쪽 `Settings > Accelerator > GPU T4`와 `Internet on` (전화번호 인증 필요)

Hugging Face 토큰은 필요 없다. 이 노트북은 공개 모델과 공개 가중치만 받는다.

## 1. 설치와 모델 다운로드

패키지를 설치한 뒤, 베이스 모델(SD 1.5, 약 2.5GB)과 IP-Adapter 다운로드를 **백그라운드로** 시작한다. 다운로드가 도는 동안 다음 셀로 넘어가도 된다.

In [ ]:
import os, sys, pathlib, subprocess, time

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ROOT = pathlib.Path("/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else ".").resolve()
WORK = ROOT / "ft-lab"
WORK.mkdir(parents=True, exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle T4 x2에서도 GPU 하나만 쓴다
print("환경:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "기타", "| 작업 폴더:", WORK)

# nvidia-smi는 환경에 따라 PATH에 없을 수 있어서 (VS Code Colab 확장 등) torch로 확인한다
import torch
if not torch.cuda.is_available():
    raise RuntimeError("torch가 GPU를 찾지 못했다. Colab은 T4 GPU 런타임(VS Code는 T4로 새 Colab 서버)으로, "
                       "Kaggle은 Accelerator를 GPU T4로 바꾸고 다시 실행한다.")
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f}GB",
      "| torch", torch.__version__)
!pip install -q diffusers==0.42.1 transformers==5.22.2 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2 ftfy tensorboard
# Colab에 미리 깔린 torchao(0.10)는 peft 0.21과 맞지 않아 LoRA를 만들 때 오류가 난다. 이 실습은 torchao를 쓰지 않는다.
!pip uninstall -q -y torchao

SD15 = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DOWNLOAD_LOG = WORK / "download.log"
DOWNLOAD = subprocess.Popen(
    "hf download " + SD15 + " --include '*.json' --include '*.txt' --include '*.fp16.safetensors'"
    " && hf download h94/IP-Adapter models/ip-adapter_sd15.bin models/image_encoder/config.json models/image_encoder/model.safetensors",
    shell=True, stdout=open(DOWNLOAD_LOG, "w"), stderr=subprocess.STDOUT,
)
print("모델 다운로드를 백그라운드에서 시작했다. 로그:", DOWNLOAD_LOG)

## 2. 학습 대상 고르기

`SUBJECT`를 하나 고른다.

| `SUBJECT` | 대상 | 장수 |
|---|---|---|
| `"can"` | 고래 그림이 있는 캔 | 6 |
| `"cat2"` | 고양이 | 5 |
| `"monster_toy"` | 몬스터 인형 | 5 |
| `"mine"` | **내 사진** | 10~20 권장 |

샘플 세 대상은 DreamBooth 논문의 공개 데이터셋([google/dreambooth](https://github.com/google/dreambooth))에서 받는다.

**내 사진을 쓸 때 (`"mine"`)**
- 같은 물건 하나를 **10~20장**, **배경과 각도를 다양하게** 찍는다. 배경이 모두 같으면 모델이 물건과 함께 배경까지 외운다.
- 물건이 사진 가운데에 크게 오게 찍는다. 이 셀이 가운데를 정사각형으로 잘라 512px로 맞춘다.
- JPG나 PNG로 올린다 (아이폰 HEIC는 미리 JPG로 바꾼다). 사람 얼굴은 피한다.
- `MY_CLASS`에 물건 종류를 영어 한 단어로 적는다 (예: `"mug"`, `"shoe"`, `"dog"`).
- Colab은 셀을 실행하면 파일 선택 창이 뜬다. Kaggle은 `Add Input`으로 사진 데이터셋을 붙이고 `KAGGLE_INPUT_DIR`에 경로를 적는다.
- 미리 학습한 Textual Inversion 임베딩은 샘플 세 대상 것만 있다. 그래서 내 사진은 LoRA만 학습하고, 네 방법 비교는 `COMPARE_SUBJECT`의 샘플 대상으로 본다.

In [ ]:
SUBJECT = "cat2"          # "can", "cat2", "monster_toy", "mine"
MY_CLASS = "mug"          # SUBJECT = "mine"일 때만: 물건 종류 (영어 한 단어)
COMPARE_SUBJECT = "cat2"  # SUBJECT = "mine"일 때만: 네 방법 비교에 쓸 샘플 대상
KAGGLE_INPUT_DIR = "/kaggle/input/my-photos"  # Kaggle에서 "mine"일 때만

import shutil, urllib.request
from PIL import Image, ImageOps

# 대상 이름: (class 단어, 장수). class는 데이터셋의 prompts_and_classes.txt를 따른다.
SAMPLES = {"can": ("can", 6), "cat2": ("cat", 5), "monster_toy": ("toy", 5)}
DREAMBOOTH_RAW = "https://raw.githubusercontent.com/google/dreambooth/main/dataset"


def download_sample(name):
    d = WORK / "data" / name
    d.mkdir(parents=True, exist_ok=True)
    for i in range(SAMPLES[name][1]):
        f = d / f"{i:02d}.jpg"
        if not f.exists():
            urllib.request.urlretrieve(f"{DREAMBOOTH_RAW}/{name}/{i:02d}.jpg", f)
    return d


def square_512(src, dst):
    im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
    s = min(im.size)
    left, top = (im.width - s) // 2, (im.height - s) // 2
    im.crop((left, top, left + s, top + s)).resize((512, 512), Image.LANCZOS).save(dst, quality=95)


if SUBJECT == "mine":
    CLASS = MY_CLASS
    TRAIN_DIR = WORK / "data" / "mine"
    shutil.rmtree(TRAIN_DIR, ignore_errors=True)
    TRAIN_DIR.mkdir(parents=True)
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()  # 여러 장을 한 번에 고른다
        sources = [pathlib.Path(n) for n in uploaded]
    else:
        sources = sorted(p for p in pathlib.Path(KAGGLE_INPUT_DIR).rglob("*")
                         if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    for i, src in enumerate(sources):
        square_512(src, TRAIN_DIR / f"{i:02d}.jpg")
    assert len(sources) >= 3, "사진이 너무 적다. 10~20장을 올린다."
else:
    assert SUBJECT in SAMPLES, f"SUBJECT는 {list(SAMPLES)} 또는 'mine' 중 하나다"
    CLASS = SAMPLES[SUBJECT][0]
    TRAIN_DIR = download_sample(SUBJECT)
    COMPARE_SUBJECT = SUBJECT

assert COMPARE_SUBJECT in SAMPLES
COMPARE_CLASS = SAMPLES[COMPARE_SUBJECT][0]
COMPARE_DIR = download_sample(COMPARE_SUBJECT)
INSTANCE_PROMPT = f"a photo of sks {CLASS}"
print(f"학습 대상: {SUBJECT} ({len(list(TRAIN_DIR.glob('*.jpg')))}장) | instance prompt: '{INSTANCE_PROMPT}'")
print(f"비교 대상: {COMPARE_SUBJECT}")

격자 함수와 학습 사진 미리보기. 학습 사진에 무엇이 공통으로 들어 있는지 눈여겨본다. 모델은 그 공통점을 배운다.

In [ ]:
from collections.abc import Sequence
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display


def make_grid(
    images: Sequence[Image.Image],
    cols: int,
    col_labels: Sequence[str] | None = None,
    row_labels: Sequence[str] | None = None,
    pad: int = 8,
    label_h: int = 28,
    label_w: int = 140,
) -> Image.Image:
    """이미지를 행 우선 순서로 cols열 격자에 붙인다.

    col_labels는 열 위쪽(예: 프롬프트), row_labels는 행 왼쪽(예: 기법 이름)에 적는다.
    """
    if not images:
        raise ValueError("images가 비어 있다")
    rows = -(-len(images) // cols)
    w, h = images[0].size
    top = label_h if col_labels else 0
    left = label_w if row_labels else 0

    grid = Image.new(
        "RGB",
        (left + cols * w + (cols + 1) * pad, top + rows * h + (rows + 1) * pad),
        "white",
    )
    draw = ImageDraw.Draw(grid)

    for i, img in enumerate(images):
        r, c = divmod(i, cols)
        x = left + pad + c * (w + pad)
        y = top + pad + r * (h + pad)
        grid.paste(img.resize((w, h)) if img.size != (w, h) else img, (x, y))

    for c, text in enumerate(col_labels or []):
        draw.text((left + pad + c * (w + pad), 6), text, fill="black")
    for r, text in enumerate(row_labels or []):
        draw.text((6, top + pad + r * (h + pad) + h // 2), text, fill="black")

    return grid


def thumbs(folder, size=192):
    return [ImageOps.fit(Image.open(p).convert("RGB"), (size, size)) for p in sorted(folder.glob("*.jpg"))]


display(make_grid(thumbs(TRAIN_DIR), cols=6))

## 3. LoRA 학습

diffusers 공식 예제 스크립트(`train_dreambooth_lora.py`, v0.42.1)로 학습한다. T4에 맞춘 설정은 다음과 같다.

| 옵션 | 값 | 이유 |
|---|---|---|
| `--resolution` | 512 | SD 1.5의 기본 해상도 |
| `--mixed_precision` | fp16 | T4는 bf16을 지원하지 않는다 |
| `--gradient_checkpointing` | 켬 | 계산을 조금 더 해서 GPU 메모리를 아낀다 |
| `--use_8bit_adam` | 켬 | optimizer 상태를 8비트로 저장해 메모리를 아낀다 |
| `--rank` | 8 | LoRA 어댑터의 크기. 결과 파일은 약 6MB다 |
| `--max_train_steps` | `STEPS` | 학습 반복 횟수 |

학습이 도는 동안에는 GPU를 다 쓰므로 다른 셀을 실행하지 않는다. 진행 막대가 끝까지 가면 완료다.

In [ ]:
STEPS = 500  # 잠정값. T4에서 10~15분에 끝나도록 실측해서 조정한다
TRAIN_SCRIPT = WORK / "train_dreambooth_lora.py"
LORA_DIR = WORK / f"lora-{SUBJECT}"

if not TRAIN_SCRIPT.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/huggingface/diffusers/v0.42.1/examples/dreambooth/train_dreambooth_lora.py",
        TRAIN_SCRIPT,
    )

print("베이스 모델 다운로드를 기다린다...")
if DOWNLOAD.wait() != 0:
    print(DOWNLOAD_LOG.read_text()[-2000:])
    raise RuntimeError("모델 다운로드 실패. 위 로그를 확인하고 1번 셀을 다시 실행한다.")
print("다운로드 완료. 학습을 시작한다.")

cmd = [
    "accelerate", "launch", "--num_processes=1", "--num_machines=1", "--mixed_precision=fp16", "--dynamo_backend=no",
    str(TRAIN_SCRIPT),
    f"--pretrained_model_name_or_path={SD15}",
    "--variant=fp16",
    f"--instance_data_dir={TRAIN_DIR}",
    f"--instance_prompt={INSTANCE_PROMPT}",
    f"--output_dir={LORA_DIR}",
    "--resolution=512",
    "--train_batch_size=1",
    "--gradient_accumulation_steps=1",
    "--gradient_checkpointing",
    "--use_8bit_adam",
    "--learning_rate=1e-4",
    "--lr_scheduler=constant",
    "--lr_warmup_steps=0",
    f"--max_train_steps={STEPS}",
    "--rank=8",
    "--mixed_precision=fp16",
    "--checkpointing_steps=100000",
    "--report_to=tensorboard",
    "--seed=0",
]
t0 = time.time()
# 출력을 그대로 흘려 보내고(진행 막대 포함), 실패하면 셀을 멈춘다
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for chunk in iter(lambda: proc.stdout.read(256), ""):
    print(chunk, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError("학습이 실패했다. 위 출력의 마지막 오류(Traceback)를 확인한다. "
                       "시간이 없으면 아래 '비상용 LoRA' 셀에서 USE_HUB_LORA = True로 진행한다.")
print(f"학습 시간: {(time.time() - t0) / 60:.1f}분")
print("결과:", *LORA_DIR.glob("*.safetensors"))

### 학습을 못 했을 때 (비상용 LoRA)

학습이 오류로 멈췄거나 시간이 부족하면 `USE_HUB_LORA = True`로 바꾸고 실행한다. 샘플 세 대상은 미리 학습한 LoRA를 Hub(`munain0417/sd15-practice-lora`)에서 받는다. 학습이 잘 끝났으면 `False` 그대로 실행한다.

In [ ]:
USE_HUB_LORA = False
LORA_HUB_REPO = "munain0417/sd15-practice-lora"
TI_HUB_REPO = "munain0417/sd15-practice-ti"

my_lora_file = LORA_DIR / "pytorch_lora_weights.safetensors"
if USE_HUB_LORA or not my_lora_file.exists():
    if SUBJECT == "mine":
        raise RuntimeError("내 사진 LoRA는 Hub에 없다. 학습 셀을 다시 실행하거나, SUBJECT를 샘플 대상으로 바꾼다.")
    print(f"Hub의 비상용 LoRA를 쓴다: {LORA_HUB_REPO}/{SUBJECT}/")
    MY_LORA = dict(pretrained_model_name_or_path_or_dict=LORA_HUB_REPO,
                   weight_name=f"{SUBJECT}/pytorch_lora_weights.safetensors")
else:
    print("직접 학습한 LoRA를 쓴다:", my_lora_file)
    MY_LORA = dict(pretrained_model_name_or_path_or_dict=str(LORA_DIR),
                   weight_name="pytorch_lora_weights.safetensors")

# 네 방법 비교에 쓸 LoRA: 샘플 대상이면 위와 같고, 내 사진이면 비교 대상의 비상용 LoRA를 쓴다
if SUBJECT == "mine":
    COMPARE_LORA = dict(pretrained_model_name_or_path_or_dict=LORA_HUB_REPO,
                        weight_name=f"{COMPARE_SUBJECT}/pytorch_lora_weights.safetensors")
else:
    COMPARE_LORA = MY_LORA

## 4. 네 방법 비교

같은 프롬프트, 같은 seed로 네 방법을 나란히 본다.

| 방법 | 학습하는 것 | 프롬프트에서 대상을 부르는 말 |
|---|---|---|
| 베이스 | 없음 | `a cat` (일반 단어) |
| **LoRA** (방금 학습) | UNet에 붙인 작은 어댑터 | `sks cat` |
| Textual Inversion | 새 단어 하나의 임베딩 벡터 (미리 학습) | `<cat2>` |
| IP-Adapter | 없음. 참조 사진 1장을 이미지 인코더로 넣는다 | `a cat` + 참조 사진 |

보는 점
- 대상이 학습 사진의 그 물건처럼 나오는가 (모양, 색, 무늬)
- 프롬프트의 배경(해변, 눈밭)과 화풍(수채화)을 따르는가, 아니면 학습 사진의 배경으로 끌려가는가

`can`의 Textual Inversion은 일부러 약한 예로 남겨 두었다. 학습 사진 6장이 모두 비슷한 겨울 강변이라, 오래 학습하면 캔보다 그 장면을 배운다. 그래서 중간(1500스텝) 임베딩을 올렸고, 캔 그림은 따라 하지만 모양과 배경 제어는 LoRA보다 약하다.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(SD15, variant="fp16", torch_dtype=torch.float16).to("cuda")
pipe.set_progress_bar_config(disable=True)

SEED = 42
SCENES = ["a photo of {} on the beach", "a photo of {} in the snow", "a watercolor painting of {}"]


def generate(subject_words, seed=SEED, **kwargs):
    """SCENES의 각 장면에 subject_words를 넣어 한 장씩 만든다 (같은 seed)."""
    return [
        pipe(scene.format(subject_words), num_inference_steps=30, guidance_scale=7.5,
             generator=torch.Generator("cuda").manual_seed(seed), **kwargs).images[0]
        for scene in SCENES
    ]


cls, token = COMPARE_CLASS, f"<{COMPARE_SUBJECT}>"
rows = {}

t0 = time.time()
rows["base"] = generate(f"a {cls}")

pipe.load_lora_weights(**COMPARE_LORA, adapter_name="compare")
rows["LoRA"] = generate(f"sks {cls}")
pipe.unload_lora_weights()

if token not in pipe.tokenizer.get_vocab():
    pipe.load_textual_inversion(TI_HUB_REPO, weight_name=f"{COMPARE_SUBJECT}/learned_embeds.safetensors")
rows["Textual Inv."] = generate(token)

reference = Image.open(sorted(COMPARE_DIR.glob("*.jpg"))[0]).convert("RGB")
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
pipe.set_ip_adapter_scale(0.6)
rows["IP-Adapter"] = generate(f"a {cls}", ip_adapter_image=reference)
pipe.unload_ip_adapter()
print(f"생성 시간: {time.time() - t0:.0f}초")

print("IP-Adapter 참조 사진:")
display(ImageOps.fit(reference, (192, 192)))
compare_grid = make_grid(
    [img for imgs in rows.values() for img in imgs], cols=len(SCENES),
    col_labels=[s.format("...") for s in SCENES], row_labels=list(rows),
)
compare_grid.save(WORK / f"compare_{COMPARE_SUBJECT}.png")
display(compare_grid)

### 내 사진 LoRA 확인 (`SUBJECT = "mine"`일 때만)

내 물건으로 학습한 LoRA를 베이스와 비교한다. 샘플 대상을 골랐으면 이 셀은 건너뛴다.

In [ ]:
if SUBJECT == "mine":
    mine_rows = {"base": generate(f"a {CLASS}")}
    pipe.load_lora_weights(**MY_LORA, adapter_name="mine")
    mine_rows["my LoRA"] = generate(f"sks {CLASS}")
    pipe.unload_lora_weights()
    display(make_grid([img for imgs in mine_rows.values() for img in imgs], cols=len(SCENES),
                      col_labels=[s.format("...") for s in SCENES], row_labels=list(mine_rows)))
else:
    print("샘플 대상을 골랐으므로 건너뛴다.")

## 5. LoRA 세기 조절과 자유 생성

LoRA는 원래 가중치에 더하는 값이라 세기(scale)를 바꿀 수 있다. 0이면 베이스와 같고, 1이면 학습한 그대로다. 세기를 낮추면 대상의 특징은 약해지지만 프롬프트의 배경이 잘 살아나는 경우가 많다.

In [ ]:
SCALES = [0.0, 0.4, 0.7, 1.0]
prompt = f"a photo of sks {CLASS} on the beach"  # 자유롭게 바꿔 본다

pipe.load_lora_weights(**MY_LORA, adapter_name="mine")
images = []
for s in SCALES:
    pipe.set_adapters(["mine"], adapter_weights=[s])
    images.append(pipe(prompt, num_inference_steps=30, guidance_scale=7.5,
                       generator=torch.Generator("cuda").manual_seed(SEED)).images[0])
display(make_grid(images, cols=len(SCALES), col_labels=[f"scale {s}" for s in SCALES]))

In [ ]:
# 자유 생성: 프롬프트, 세기, seed를 바꿔 본다. LoRA는 위 셀에서 붙인 상태다.
prompt = f"a photo of sks {CLASS} on a wooden table, studio lighting"
LORA_SCALE = 0.8

pipe.set_adapters(["mine"], adapter_weights=[LORA_SCALE])
images = [pipe(prompt, num_inference_steps=30, guidance_scale=7.5,
               generator=torch.Generator("cuda").manual_seed(seed)).images[0] for seed in [1, 2, 3]]
display(make_grid(images, cols=3, col_labels=["seed 1", "seed 2", "seed 3"]))

## 정리

- **LoRA**는 모델 본체를 고정하고 작은 어댑터(약 6MB)만 학습한다. 적은 사진으로 대상을 가장 잘 따라 한다.
- **Textual Inversion**은 단어 하나의 벡터(약 3KB)만 학습한다. 가볍지만 모델이 원래 그릴 수 있는 범위 안에서만 대상을 표현한다.
- **IP-Adapter**는 학습 없이 참조 사진 한 장으로 비슷한 이미지를 만든다. 빠르지만 대상의 세부를 정확히 지키지는 못한다.
- 세 방법 모두 **학습 사진의 배경이 한 가지뿐이면 배경까지 배운다.** 내 사진으로 다시 해 볼 때는 배경과 각도를 다양하게 찍는다.

(선택) 대상 대신 한 화가의 그림이나 같은 화풍의 이미지 10~20장으로 같은 코드를 돌리면 화풍 LoRA가 된다. `MY_CLASS`는 `"painting"`처럼 적는다.